In [9]:
import pandas as pd
import numpy as np
import time
from datetime import datetime


StatementMeta(, 7478bf46-c514-41a5-8a1c-7902824d5ef6, 11, Finished, Available, Finished)

In [10]:
reviews = pd.read_csv("/lakehouse/default/Files/olist_order_reviews_dataset.csv")
print(reviews.head())

StatementMeta(, 7478bf46-c514-41a5-8a1c-7902824d5ef6, 12, Finished, Available, Finished)

                          review_id                          order_id  \
0  7bc2406110b926393aa56f80a40eba40  73fc7af87114b39712e6da79b0a377eb   
1  80e641a11e56f04c1ad469d5645fdfde  a548910a1c6147796b98fdf73dbeba33   
2  228ce5500dc1d8e020d8d1322874b6f0  f9e4b658b201a9f2ecdecbb34bed034b   
3  e64fb393e7b32834bb789ff8bb30750e  658677c97b385a9be170737859d3511b   
4  f7c4243c7fe1938f181bec41a392bdeb  8e6bfb81e283fa7e4f11123a3fb894f1   

   review_score review_comment_title  \
0             4                  NaN   
1             5                  NaN   
2             5                  NaN   
3             5                  NaN   
4             5                  NaN   

                              review_comment_message review_creation_date  \
0                                                NaN  2018-01-18 00:00:00   
1                                                NaN  2018-03-10 00:00:00   
2                                                NaN  2018-02-17 00:00:00   
3           

In [11]:
profile = pd.DataFrame({
    'Column': reviews.columns.values,
    'negative(%)': [
        len(reviews[col][reviews[col] < 0]) / len(reviews) * 100 if col in reviews.select_dtypes(include=[np.number]).columns else 0
        for col in reviews.columns
    ],  
    'zero(%)': [
        len(reviews[col][reviews[col] == 0]) / len(reviews) * 100 if col in reviews.select_dtypes(include=[np.number]).columns else 0
        for col in reviews.columns
    ],  
    'duplicates': reviews.duplicated().sum(), 
    'unique': reviews.nunique().values, 
})

profile

StatementMeta(, 7478bf46-c514-41a5-8a1c-7902824d5ef6, 13, Finished, Available, Finished)

,Column,negative(%),zero(%),duplicates,unique
0,review_id,0.0,0.0,0,98410
1,order_id,0.0,0.0,0,98673
2,review_score,0.0,0.0,0,5
3,review_comment_title,0.0,0.0,0,4527
4,review_comment_message,0.0,0.0,0,36159
5,review_creation_date,0.0,0.0,0,636
6,review_answer_timestamp,0.0,0.0,0,98248


In [13]:
# There are no null/negative values and there are no duplicates
# The title and comment columns contain many null values but there are no review scores with null values


reviews['review_score']=reviews['review_score'].astype(int)
reviews['review_creation_date'] = pd.to_datetime(reviews['review_creation_date'])
reviews['review_answer_timestamp'] = pd.to_datetime(reviews['review_answer_timestamp'])


for col in reviews.columns:
    reviews[col] = reviews[col].astype(str)
    reviews[col] = reviews[col].str.replace('\xa0', ' ', regex=True)
    reviews[col] = reviews[col].str.replace('[\x00-\x1f\x7f-\x9f]', '', regex=True)
    reviews[col] = reviews[col].str.replace('\r', ' ', regex=True)
    reviews[col] = reviews[col].str.replace('\n', ' ', regex=True)
    reviews[col] = reviews[col].str.strip()

reviews_cleaned = reviews

StatementMeta(, 7478bf46-c514-41a5-8a1c-7902824d5ef6, 15, Finished, Available, Finished)

In [ ]:
from pandas.api.types import is_datetime64_any_dtype

# Check post-cleaned dataframe

assert reviews['order_id'].isnull().sum() == 0, "Null values found in order id"
assert reviews['review_id'].isnull().sum() == 0, "Null values found in review id"
assert reviews['review_score'].isnull().sum() == 0, "Null values found in review score"

assert is_datetime64_any_dtype(reviews['review_creation_date']), "Column 'shipping_limit_date' is not of datetime type"

columns_to_check = ['order_id', 'review_id', 'seller_id', 'review_score']

# Check if all values in each of these columns are non-zero
assert reviews[columns_to_check].ne(0).all().all(), "Some values are zero in the specified columns"